# Auto Loader — Live Demo: Continuous Ingestion from ADLS
### GlobalMart Data Engineering · College Presentation Notebook

---

## What We Are Proving

> **Auto Loader does NOT re-read files it has already processed.**  
> **When a NEW file lands in ADLS, Auto Loader detects it automatically — zero manual intervention.**

---

## Live Demo Run Order

| # | Phase | Action | Expected Result |
|---|-------|--------|-----------------|
| 1 | **Setup** | Run setup cell | ADLS connected, all paths defined |
| 2 | **Phase 1** | Load orders.csv with `availableNow=True` | Bronze created, ~1.4M rows, stream stops |
| 3 | **Phase 2** | Create `orders_new.csv` → save to staging | New file ready, NOT yet in `raw/` |
| 4 | **Phase 3** | Start continuous stream (same checkpoint) | Stream idle — sees no new files |
| 5 | **Phase 4** | Copy `orders_new.csv` into `raw/` | Stream detects it in ≤30 sec, count jumps |
| 6 | **Validation** | No-duplicate proof, checkpoint inspection | Exactly-once processing confirmed |

---

## Auto Loader vs Manual Approach

| Feature | `spark.read` (manual) | Auto Loader (`cloudFiles`) |
|---------|----------------------|--------------------------|
| Tracks processed files | No — re-reads everything | Yes — checkpoint remembers every file |
| Schema inference | Manual every run | Automatic, stored, reused |
| New file detection | Must re-trigger entire job | Detects automatically on next interval |
| Duplicate risk on re-run | High | Zero |
| Production grade | No | Yes |

---
## How Auto Loader Tracks Files — The Checkpoint

```
ADLS raw/ folder:
  orders.csv          ← Phase 1: processed ✅  (checkpoint remembers this)
  orders_new.csv      ← Phase 4: we drop this  🔴 (not yet seen)

Checkpoint folder (_checkpoints/orders_demo/):
  sources/   ← records EVERY processed file (path + size + mtime)
  offsets/   ← current stream reading position
  commits/   ← confirmed completed micro-batch IDs

What happens every trigger (30 seconds):
  1. List all files in raw/
  2. Compare against checkpoint sources/
  3. orders.csv      → already in checkpoint → SKIP
  4. orders_new.csv  → NOT in checkpoint     → PROCESS ← this is the moment
  5. Write 5 new rows to Bronze Delta
  6. Update checkpoint — orders_new.csv now recorded

Next trigger: both files in checkpoint → nothing to do → stream idles
```

> **Each file is processed exactly once — this is exactly-once delivery.**

In [ ]:
# ─── SETUP — Run this cell FIRST before anything else ─────────────────────────
# WARNING: Do NOT push this notebook to GitHub with the real key

storage_account_name = "amazonprojectadls"
container_name       = "amazon-data"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"  # ← replace before running

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

base_path       = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"
raw_path        = f"{base_path}/raw"
bronze_path     = f"{base_path}/bronze"
staging_path    = f"{base_path}/staging"
checkpoint_path = f"{base_path}/_checkpoints/orders_demo"
output_path     = f"{bronze_path}/orders_demo_stream"
schema_path     = f"{base_path}/_autoloader/schemas/orders_demo"

print("Setup complete!")
print(f"  raw_path    : {raw_path}")
print(f"  output_path : {output_path}")
print(f"  checkpoint  : {checkpoint_path}")

---
## Phase 1 — Initial Full Load

**Trigger:** `availableNow=True` — process all files currently in `raw/`, then stop.  
**What happens after:** checkpoint records `orders.csv` as processed. Stream stops.  
**Expected result:** ~1.4 million rows in `bronze/orders_demo_stream`.

In [ ]:
# ─── Check what files are currently in raw/ ────────────────────────────────────
print("Files in ADLS raw/ folder (Auto Loader will process ALL of these):")
print("-" * 60)

csv_files = [f for f in dbutils.fs.ls(raw_path) if f.name.endswith(".csv")]
for f in csv_files:
    print(f"  {f.name:<35} {f.size / 1024 / 1024:>8.2f} MB")

print(f"\nTotal CSV files found: {len(csv_files)}")

In [ ]:
# ─── CLEAN START — run this if you get schema/checkpoint errors ────────────────
# Safe to run even on first run — silently skips if nothing exists yet

for path, label in [
    (output_path,     "Bronze output table"),
    (checkpoint_path, "Checkpoint folder"),
    (schema_path,     "Auto Loader schema cache"),
]:
    try:
        dbutils.fs.rm(path, recurse=True)
        print(f"  Deleted : {label}")
    except:
        print(f"  Skipped : {label} (did not exist)")

print()
print("Clean start ready — run Phase 1 next.")

In [ ]:
# ─── PHASE 1: Load all current files with availableNow=True ───────────────────
# delta.columnMapping.mode = name  → allows special characters in column names
# e.g. shipping_tier.csv has "Cost (Rs)" — parentheses are invalid without mapping

from pyspark.sql.functions import current_timestamp, input_file_name

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header",      "true")
    .option("inferSchema", "true")
    .load(raw_path)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
)
for c in raw_stream.columns:
    clean_name = (
        c.replace(" ", "_")
         .replace("(", "")
         .replace(")", "")
         .replace(".", "")
         .replace("%", "Pct")
         .replace("-", "_")
    )
    raw_stream = raw_stream.withColumnRenamed(c, clean_name)
phase1_query = (
    raw_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation",          checkpoint_path)
    .option("mergeSchema",                 "true")
    .option("delta.columnMapping.mode",    "name")
    .option("delta.minReaderVersion",      "2")
    .option("delta.minWriterVersion",      "5")
    .trigger(availableNow=True)
    .start(output_path)
)

phase1_query.awaitTermination()
print("Phase 1 complete — all current files processed, stream stopped.")

In [ ]:
# ─── Phase 1 Results ──────────────────────────────────────────────────────────
from pyspark.sql.functions import count, min as spark_min, max as spark_max

bronze_df    = spark.read.format("delta").load(output_path)
phase1_count = bronze_df.count()

print("=" * 60)
print("  PHASE 1 RESULTS")
print("=" * 60)
print(f"  Total rows in Bronze : {phase1_count:,}")
print()
print("Rows per source file:")
(
    bronze_df
    .groupBy("_source_file")
    .agg(
        count("*").alias("row_count"),
        spark_min("_ingested_at").alias("first_ingested"),
        spark_max("_ingested_at").alias("last_ingested")
    )
    .show(truncate=False)
)

print(f"Baseline saved: phase1_count = {phase1_count:,}")
print("This number must NOT change until we drop orders_new.csv in Phase 4.")

In [ ]:
# ─── Lock in column mapping on Bronze table ────────────────────────────────────
# shipping_tier.csv has "Cost (Rs)" — parentheses/spaces are invalid in Delta
# unless column mapping mode = name is set as a TABLE PROPERTY (not just a write option).
# Setting it here ensures Phase 3 stream can start without the DELTA_INVALID_CHARACTERS error.

spark.sql(f"""
    ALTER TABLE delta.`{output_path}`
    SET TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name',
        'delta.minReaderVersion'   = '2',
        'delta.minWriterVersion'   = '5'
    )
""")
print("Column mapping properties set on Bronze table.")
print("Phase 3 continuous stream will now start without DELTA_INVALID_CHARACTERS errors.")

In [ ]:
# ─── Sample rows with metadata columns ────────────────────────────────────────
print("Sample Bronze rows — notice _source_file and _ingested_at columns:")
print()

(
    spark.read.format("delta").load(output_path)
    .select("OrderID", "CustomerID", "OrderChannel", "OrderDate", "_ingested_at", "_source_file")
    .limit(5)
    .show(truncate=False)
)

print("After Phase 4, rows from orders_new.csv will show a DIFFERENT _source_file.")
print("That is how we prove which file each row came from.")

---
## Phase 2 — Prepare the New File (Staging)

We create `orders_new.csv` with 5 fresh GlobalMart orders and write it directly to the **staging** folder using `dbutils.fs.put()`.
It is **not** in `raw/` yet — Auto Loader cannot see it.

> Staging = holding area. We control exactly when Auto Loader sees the file by choosing when to copy it into `raw/`.

In [ ]:
# ─── PHASE 2: Write orders_new.csv directly to staging via dbutils ─────────────
# dbutils.fs.put() writes the file into ADLS using the configured storage key.
# No manual Azure Portal upload needed — avoids the ADLS Gen2 directory issue.

orders_new_content = """\
OrderID,CustomerID,OrderDate,ShippingDate,ExpectedDeliveryDate,ActualDeliveryDate,ShippingTierID,SupplierID,OrderChannel
ORD-NEW-2026-001,CUST-08966,2026-06-15,2026-06-16,2026-06-20,,SHP-001,SUP-01,Online
ORD-NEW-2026-002,CUST-14521,2026-06-15,2026-06-16,2026-06-21,,SHP-002,SUP-02,Mobile App
ORD-NEW-2026-003,CUST-29834,2026-06-15,2026-06-17,2026-06-22,,SHP-001,SUP-03,Online
ORD-NEW-2026-004,CUST-07312,2026-06-15,2026-06-16,2026-06-20,,SHP-003,SUP-01,In-Store
ORD-NEW-2026-005,CUST-35671,2026-06-15,2026-06-18,2026-06-23,,SHP-002,SUP-04,Mobile App"""

staged_csv = f"{staging_path}/orders_new.csv"
dbutils.fs.put(staged_csv, orders_new_content, overwrite=True)

print(f"File written to staging: {staged_csv}")
print()

# Verify it is readable
preview_df = spark.read.option("header", "true").csv(staged_csv)
print("Preview:")
preview_df.show(truncate=False)
print(f"Row count : {preview_df.count()}")
print()
print("File is in staging — NOT yet visible to Auto Loader (watching raw/ only).")

In [ ]:
# ─── Confirm file locations before Phase 3 ────────────────────────────────────
raw_csv_names     = [f.name for f in dbutils.fs.ls(raw_path)     if f.name.endswith(".csv")]
staging_csv_names = [f.name for f in dbutils.fs.ls(staging_path) if f.name.endswith(".csv")]

print("raw/ folder (Auto Loader watches this):")
for n in raw_csv_names:
    print(f"  {n}")
print(f"  orders_new.csv present in raw/     : {'orders_new.csv' in raw_csv_names}")

print()
print("staging/ folder (holding area):")
for n in staging_csv_names:
    print(f"  {n}")
print(f"  orders_new.csv present in staging/ : {'orders_new.csv' in staging_csv_names}")

print()
print("CONFIRMED: orders_new.csv is in staging only. Auto Loader is unaware of it.")

---
## Phase 3 — Start Continuous Stream

We start Auto Loader with `trigger(processingTime='30 seconds')`.  
It uses the **same checkpoint** as Phase 1 — so it already knows `orders.csv` was processed.

```
Phase 1 checkpoint state:  orders.csv → DONE
Phase 3 first trigger:     orders.csv → SKIP (in checkpoint), nothing else → IDLE
Phase 4 (file dropped):    orders_new.csv → NEW → PROCESS
```

> The stream runs **in the background** after the start cell.  
> You can run subsequent cells while it is active.  
> Stop it with `continuous_query.stop()` when the demo ends.

In [ ]:
# ─── PHASE 3: Start continuous stream (same checkpoint as Phase 1) ─────────────
from pyspark.sql.functions import current_timestamp, input_file_name

continuous_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header",      "true")
    .option("inferSchema", "true")
    .load(raw_path)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

continuous_query = (
    continuous_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation",          checkpoint_path)
    .option("mergeSchema",                 "true")
    .option("delta.columnMapping.mode",    "name")
    .option("delta.minReaderVersion",      "2")
    .option("delta.minWriterVersion",      "5")
    .trigger(processingTime="30 seconds")
    .start(output_path)
)

print("Continuous stream started!")
print(f"  Stream ID  : {continuous_query.id}")
print(f"  Is active  : {continuous_query.isActive}")
print(f"  Trigger    : every 30 seconds")
print(f"  Watching   : {raw_path}")
print(f"  Output     : {output_path}")
print()
print("Stream runs in the background. Continue to the next cell.")

In [ ]:
# ─── Confirm: count unchanged (no new files yet) ───────────────────────────────
import time
time.sleep(5)  # let stream settle after start

current_count = spark.read.format("delta").load(output_path).count()

print("Stream status BEFORE dropping new file:")
print(f"  Phase 1 baseline : {phase1_count:,}")
print(f"  Current count    : {current_count:,}")
print(f"  New rows added   : {current_count - phase1_count:,}")
print()
print("Expected: 0 new rows — stream is idle, waiting for a new file.")
print("orders.csv is in checkpoint → stream skips it every trigger.")

---
## Phase 4 — THE DEMO MOMENT

### Drop `orders_new.csv` into `raw/`

```
BEFORE:  raw/orders.csv only  → stream sees nothing new → count stays at 1.4M

ACTION:  Copy orders_new.csv from staging/ into raw/
         Next trigger fires (within 30 seconds)
         Auto Loader detects new file → ingests 5 rows

AFTER:   count = 1.4M + 5
         _source_file = orders_new.csv for those 5 rows
```

**Run the drop cell first, then the watcher cell.**

In [ ]:
# ─── DROP THE FILE — this is the demo moment ──────────────────────────────────
print("=" * 65)
print("  DROPPING orders_new.csv INTO ADLS raw/ FOLDER")
print("=" * 65)

dbutils.fs.cp(
    f"{staging_path}/orders_new.csv",   # source — prepared in Phase 2
    f"{raw_path}/orders_new.csv"         # destination — Auto Loader watches here
)

print()
print(f"File dropped: {raw_path}/orders_new.csv")
print()
print("Files now in raw/:")
for f in dbutils.fs.ls(raw_path):
    if f.name.endswith(".csv"):
        print(f"  {f.name}")
print()
print("Auto Loader trigger fires every 30 seconds.")
print("Run the next cell to watch the row count update.")

In [ ]:
# ─── WATCH the row count update in real time ──────────────────────────────────
import time

print(f"Baseline (Phase 1): {phase1_count:,} rows")
print("Checking every 15 seconds for up to 3 minutes...")
print()

detected = False
for i in range(12):
    count_now = spark.read.format("delta").load(output_path).count()
    new_rows  = count_now - phase1_count
    elapsed   = i * 15
    status    = "NEW FILE DETECTED!" if new_rows > 0 else "waiting..."

    print(f"  [{elapsed:>3}s]  Total: {count_now:>12,}  |  New rows: {new_rows:>5}  |  {status}")

    if new_rows > 0 and not detected:
        detected = True
        print()
        print("=" * 65)
        print(f"  AUTO LOADER DETECTED AND INGESTED orders_new.csv")
        print(f"  {new_rows} new rows added automatically")
        print("=" * 65)
        break

    time.sleep(15)

if not detected:
    print()
    print("Not detected within 3 minutes.")
    print(f"Check stream is running: continuous_query.isActive = {continuous_query.isActive}")

In [ ]:
# ─── Show rows by source file after Phase 4 ───────────────────────────────────
from pyspark.sql.functions import count, min as spark_min, max as spark_max

print("Bronze table breakdown by source file:")
(
    spark.read.format("delta").load(output_path)
    .groupBy("_source_file")
    .agg(
        count("*").alias("row_count"),
        spark_min("_ingested_at").alias("ingested_from"),
        spark_max("_ingested_at").alias("ingested_to")
    )
    .orderBy("ingested_from")
    .show(truncate=False)
)
print("The _ingested_at timestamps prove they were ingested at different times.")

In [ ]:
# ─── Show the 5 new rows specifically ─────────────────────────────────────────
from pyspark.sql.functions import col

print("The 5 new orders ingested automatically from orders_new.csv:")
(
    spark.read.format("delta").load(output_path)
    .filter(col("_source_file").contains("orders_new"))
    .select("OrderID", "CustomerID", "OrderChannel", "OrderDate", "_ingested_at", "_source_file")
    .show(truncate=False)
)
print("These rows did not exist in Bronze before Phase 4.")
print("No manual trigger — Auto Loader detected and ingested them automatically.")

---
## Validation — Proving Correctness

| Claim | Validation Cell | Expected Result |
|-------|----------------|-----------------|
| No duplicates | al-21 | `COUNT(*) == COUNT(DISTINCT OrderID)` |
| orders.csv not re-processed | al-22 | Ingest timestamps all from Phase 1 |
| Checkpoint records processed files | al-23 | `sources/` folder lists both files |
| Delta history shows two writes | al-24 | Version 0 = Phase 1, Version N = Phase 4 |

In [ ]:
# ─── VALIDATION 1: Zero Duplicates ────────────────────────────────────────────
bronze_df       = spark.read.format("delta").load(output_path)
total_rows      = bronze_df.count()
distinct_orders = bronze_df.select("OrderID").distinct().count()
duplicates      = total_rows - distinct_orders

print("DUPLICATE CHECK:")
print(f"  Total rows in Bronze     : {total_rows:,}")
print(f"  Distinct OrderIDs        : {distinct_orders:,}")
print(f"  Duplicates               : {duplicates:,}")
print()

if duplicates == 0:
    print("PASSED: Zero duplicates.")
    print("Auto Loader processed each order exactly once despite two stream runs.")
else:
    print(f"WARNING: {duplicates:,} duplicates found — investigate checkpoint.")

In [ ]:
# ─── VALIDATION 2: orders.csv was NOT re-processed in Phase 3/4 ───────────────
from pyspark.sql.functions import col, min as spark_min, max as spark_max

# All rows from orders.csv should share the same narrow ingest window (Phase 1)
# If they were re-ingested, we'd see a second cluster of timestamps from Phase 4 time
historical_window = (
    spark.read.format("delta").load(output_path)
    .filter(col("_source_file").contains("orders.csv"))
    .agg(
        spark_min("_ingested_at").alias("earliest"),
        spark_max("_ingested_at").alias("latest")
    )
    .first()
)

new_file_window = (
    spark.read.format("delta").load(output_path)
    .filter(col("_source_file").contains("orders_new"))
    .agg(
        spark_min("_ingested_at").alias("earliest"),
        spark_max("_ingested_at").alias("latest")
    )
    .first()
)

print("INGEST TIMELINE:")
print(f"  orders.csv     : {historical_window['earliest']}  →  {historical_window['latest']}")
print(f"  orders_new.csv : {new_file_window['earliest']}  →  {new_file_window['latest']}")
print()
print("orders.csv timestamps are all from Phase 1.")
print("orders_new.csv timestamps are from Phase 4 — LATER.")
print("This proves orders.csv was processed exactly once — checkpoint worked.")

In [ ]:
# ─── VALIDATION 3: Checkpoint Folder Inspection ────────────────────────────────
def describe_checkpoint_item(name):
    d = {
        "metadata":  "stream query ID + configuration",
        "sources/":  "LIST of every file already processed",
        "offsets/":  "current reading position in the stream",
        "commits/":  "confirmed completed micro-batch IDs",
        "state/":    "aggregation state (GROUP BY queries only)",
    }
    return d.get(name, "checkpoint metadata")

print(f"CHECKPOINT FOLDER: {checkpoint_path}")
print()
for item in dbutils.fs.ls(checkpoint_path):
    print(f"  {item.name:<20} ← {describe_checkpoint_item(item.name)}")

print()
print("The 'sources/' folder is the key to exactly-once processing.")
print("It lists every file path Auto Loader has already ingested.")
print("Any file found here will NEVER be re-processed.")

In [ ]:
# ─── VALIDATION 4: Delta Table Version History ─────────────────────────────────
bronze_df.createOrReplaceTempView("orders_demo_bronze")

print("DELTA WRITE HISTORY (one version per micro-batch):")
(
    spark.sql("DESCRIBE HISTORY orders_demo_bronze")
    .select("version", "timestamp", "operation", "operationMetrics")
    .show(10, truncate=False)
)
print("Version 0 = Phase 1 (orders.csv ingested)")
print("Last version = Phase 4 (orders_new.csv ingested — later timestamp)")

In [ ]:
# ─── Stop the stream + Final Summary ──────────────────────────────────────────
if continuous_query.isActive:
    continuous_query.stop()
    print("Continuous stream stopped.")
else:
    print("Stream already stopped.")

final_df     = spark.read.format("delta").load(output_path)
final_count  = final_df.count()
final_files  = final_df.select("_source_file").distinct().count()
dupes        = final_count - final_df.select("OrderID").distinct().count()

print()
print("=" * 60)
print("  DEMO COMPLETE — FINAL SUMMARY")
print("=" * 60)
print(f"  Total rows in Bronze     : {final_count:,}")
print(f"  Distinct source files    : {final_files}")
print(f"  Duplicate rows           : {dupes}")
print()
print("What was proved:")
print("  1. orders.csv ingested in Phase 1 — checkpoint recorded it")
print("  2. Continuous stream started — idle (orders.csv already in checkpoint)")
print("  3. orders_new.csv dropped into raw/ — detected within 30 seconds")
print("  4. 5 new rows added — orders.csv was NOT re-processed")
print("  5. Zero duplicates — exactly-once processing confirmed")

---
## Viva Q&A

### Q: What is the main advantage of Auto Loader over `spark.read`?
**A:** `spark.read` has no memory — it re-reads ALL files on every run, causing duplicates.  
Auto Loader uses a checkpoint to record every processed file. On the next run it only reads NEW files — cheaper, faster, no duplicates.

---

### Q: What is in the checkpoint folder? Where is it stored?
**A:** `sources/` (list of all processed files), `offsets/` (read position), `commits/` (completed batch IDs).  
Stored in ADLS: `abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/_checkpoints/orders_demo`

---

### Q: How does Auto Loader detect new files?
**A:** Every trigger interval it lists all files in the watched folder.  
It compares this list against `sources/` in the checkpoint.  
Files not in `sources/` → process. Files already in `sources/` → skip.

---

### Q: What is the difference between `availableNow=True` and `processingTime`?
**A:** `availableNow=True` → one-shot batch: process all pending files, then **stop**.  
`processingTime='30 seconds'` → continuous: check every 30 sec, keep running indefinitely.

---

### Q: How do we prove no duplicates?
**A:** `COUNT(*) == COUNT(DISTINCT OrderID)`. If equal → zero duplicates. See cell al-21.

---

### Q: What does `_source_file` tell us?
**A:** Added by `input_file_name()` during ingestion. It stores the full ADLS path of the source file.  
Lets us trace any Bronze row back to the exact file it came from — critical for auditing.

---

### Q: What is `schemaEvolutionMode = addNewColumns`?
**A:** If a new file arrives with an extra column, Auto Loader adds it to the schema automatically.  
Existing rows get `null` for that column. Without this mode, a schema mismatch would fail the stream.

---

### Q: Why `mergeSchema = true` on writeStream?
**A:** If the Bronze Delta table already exists and the incoming schema has a new column (from Auto Loader schema evolution), `mergeSchema=true` tells Delta to accept the new column rather than rejecting the write.

---

## Architecture — End to End

```
ADLS raw/
  orders.csv          → Phase 1: 1.4M rows ingested
  orders_new.csv      → Phase 4: 5 rows detected + ingested automatically
        |
        ▼  cloudFiles (Auto Loader)
  Schema stored  → _autoloader/schemas/orders_demo/
  Files tracked  → _checkpoints/orders_demo/sources/
        |
        ▼  writeStream → Delta
  bronze/orders_demo_stream/
    All rows from both files
    _source_file  = which file this row came from
    _ingested_at  = when Auto Loader ingested it
```